In [5]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
import pandas as pd
import numpy as np
import os

# ==================== ПАРАМЕТРЫ ====================
DATA_ROOT = "final_data"
RANDOM_STATE = 42
TRAIN_RATIO = 0.7
VAL_RATIO = 0.15
TEST_RATIO = 0.15

class TinyGraphSAGE(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels=16):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.lin = torch.nn.Linear(hidden_channels, 1)
    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.lin(x)
        return x.squeeze()  # Важно: убираем лишнюю размерность

def split_nodes(y, train_ratio, val_ratio, test_ratio, random_state):
    np.random.seed(random_state)
    n = len(y)
    perm = np.random.permutation(n)
    train_end = int(n * train_ratio)
    val_end = int(n * (train_ratio + val_ratio))
    return perm[:train_end], perm[train_end:val_end], perm[val_end:]

# ------------------- ЗАГРУЗЧИКИ -------------------
def load_elliptic_fast():
    feat_path = os.path.join(DATA_ROOT, "Elliptic", "elliptic_txs_features.csv")
    edge_path = os.path.join(DATA_ROOT, "Elliptic", "elliptic_txs_edgelist.csv")
    class_path = os.path.join(DATA_ROOT, "Elliptic", "elliptic_txs_classes.csv")

    df_feat = pd.read_csv(feat_path, header=None, low_memory=False)
    # Столбцы: 0: txId (int), 1: time_step (float), 2..: признаки
    tx_ids = df_feat[0].astype(str).values
    X = df_feat.iloc[:, 2:].values.astype(np.float32)
    # Метки
    df_class = pd.read_csv(class_path)
    df_class['label'] = df_class['class'].map({'1':0, '2':1, 'unknown':-1})
    df_class = df_class[df_class['label'] != -1]
    df_class['txId'] = df_class['txId'].astype(str)
    label_dict = df_class.set_index('txId')['label'].to_dict()
    # Отбор узлов с метками
    keep = np.array([tid in label_dict for tid in tx_ids])
    X = X[keep]
    tx_ids_keep = tx_ids[keep]
    y = np.array([label_dict[tid] for tid in tx_ids_keep], dtype=np.int64)
    # Рёбра
    edges = pd.read_csv(edge_path, header=None).values
    idx_map = {tid: i for i, tid in enumerate(tx_ids_keep)}
    edge_list = []
    for src, dst in edges:
        src_str, dst_str = str(src), str(dst)
        if src_str in idx_map and dst_str in idx_map:
            edge_list.append([idx_map[src_str], idx_map[dst_str]])
    edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous() if edge_list else torch.empty((2,0), dtype=torch.long)
    x = torch.tensor(X, dtype=torch.float32)
    y = torch.tensor(y, dtype=torch.long)
    return x, edge_index, y

def load_amazon_fast():
    feat_path = os.path.join(DATA_ROOT, "Amazon", "Amazon_full.csv")
    edge_path = os.path.join(DATA_ROOT, "Amazon", "Amazon_net_uvu_edges.csv")
    df_feat = pd.read_csv(feat_path, header=None, skiprows=1, low_memory=False)
    x = torch.tensor(df_feat.iloc[:, :-1].values.astype(np.float32))
    y = torch.tensor(df_feat.iloc[:, -1].values.astype(np.int64))
    edges = pd.read_csv(edge_path)[['row', 'col']].values
    edge_index = torch.tensor(edges.T, dtype=torch.long)
    return x, edge_index, y

def load_yelp_fast():
    feat_path = os.path.join(DATA_ROOT, "YelpChi", "YelpChi_full.csv")
    edge_path = os.path.join(DATA_ROOT, "YelpChi", "YelpChi_net_rur_edges.csv")
    df_feat = pd.read_csv(feat_path)
    x = torch.tensor(df_feat.iloc[:, :-1].values.astype(np.float32))
    y = torch.tensor(df_feat.iloc[:, -1].values.astype(np.int64))
    edges = pd.read_csv(edge_path)[['row', 'col']].values
    edge_index = torch.tensor(edges.T, dtype=torch.long)
    return x, edge_index, y

def load_mgtab_fast():
    edge_path = os.path.join(DATA_ROOT, "MGTAB", "edge_index.csv")
    label_path = os.path.join(DATA_ROOT, "MGTAB", "labels_stance.csv")

    # Читаем рёбра: файл имеет 3 строки и много столбцов
    df_edges = pd.read_csv(edge_path, header=None)
    # Вторая строка (индекс 1) — source, третья строка (индекс 2) — target
    sources = df_edges.iloc[1].values.astype(np.int64)
    targets = df_edges.iloc[2].values.astype(np.int64)
    edge_index = torch.tensor([sources, targets], dtype=torch.long)  # (2, E)

    # Загружаем метки (один столбец, 10200 строк)
    df_labels = pd.read_csv(label_path, header=None)
    y = torch.tensor(df_labels[0].values, dtype=torch.long)

    # Количество узлов определяем по максимальному индексу в рёбрах
    num_nodes = edge_index.max().item() + 1
    # Признаки: единичный вектор (можно потом заменить на степени узлов)
    x = torch.ones((num_nodes, 1), dtype=torch.float32)

    # Обрезаем метки до количества узлов (если меток меньше, остальное игнорируем)
    if len(y) > num_nodes:
        y = y[:num_nodes]

    return x, edge_index, y

# ------------------- ТЕСТ -------------------
device = torch.device('cpu')
datasets = {
    "Elliptic (Bitcoin)": load_elliptic_fast,
    "Amazon Reviews": load_amazon_fast,
    "Yelp Reviews": load_yelp_fast,
    "MGTAB": load_mgtab_fast,
}

for name, loader in datasets.items():
    print(f"\nТестируем {name}...")
    try:
        x, edge_index, y = loader()
        print(f"  Узлов: {x.shape[0]}, признаков: {x.shape[1]}, рёбер: {edge_index.shape[1]}")
        # Удалим узлы с меткой -1 (если есть)
        mask = y != -1
        x = x[mask]
        y = y[mask]
        # Обновляем рёбра: нужно переиндексировать после фильтрации (для простоты пропустим, т.к. тест только для forward без графа)
        # Но для MGTAB если много узлов без меток, лучше не удалять. Для быстрого теста просто используем весь граф, но убираем потерю.
        # Фактически проверяем только проход, необязательно удалять узлы.
        # Разбиение
        train_idx, val_idx, test_idx = split_nodes(y, TRAIN_RATIO, VAL_RATIO, TEST_RATIO, RANDOM_STATE)
        # Модель
        model = TinyGraphSAGE(x.shape[1], hidden_channels=8).to(device)
        x = x.to(device)
        edge_index = edge_index.to(device)
        y = y.to(device)
        out = model(x, edge_index)  # форма (N,)
        loss = F.binary_cross_entropy_with_logits(out[train_idx], y[train_idx].float())
        print(f"  Forward pass успешен, loss = {loss.item():.4f}")
        print(f"  ✅ {name} OK")
    except Exception as e:
        print(f"  ❌ Ошибка: {e}")


Тестируем Elliptic (Bitcoin)...
  Узлов: 46564, признаков: 165, рёбер: 36624
  Forward pass успешен, loss = 0.8031
  ✅ Elliptic (Bitcoin) OK

Тестируем Amazon Reviews...
  Узлов: 11944, признаков: 25, рёбер: 2073474
  Forward pass успешен, loss = 2.1919
  ✅ Amazon Reviews OK

Тестируем Yelp Reviews...
  Узлов: 45954, признаков: 32, рёбер: 98630
  Forward pass успешен, loss = 0.8103
  ✅ Yelp Reviews OK

Тестируем MGTAB...
  Узлов: 10199, признаков: 1, рёбер: 1700108
  Forward pass успешен, loss = 0.6544
  ✅ MGTAB OK


/var/folders/md/78mwsg9n23525_j8040kg6fw0000gn/T/ipykernel_5623/1794423335.py:97: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:256.)
  edge_index = torch.tensor([sources, targets], dtype=torch.long)  # (2, E)


In [4]:
import pandas as pd
edges_sample = pd.read_csv("data/MGTAB/edge_index.csv", header=None, nrows=5)
print("Рёбра (первые 5 строк):\n", edges_sample)
labels_sample = pd.read_csv("data/MGTAB/labels_stance.csv", header=None, nrows=5)
print("Метки (первые 5 строк):\n", labels_sample)
print("Всего строк в edge_index.csv:", sum(1 for _ in open("data/MGTAB/edge_index.csv")))
print("Всего строк в labels_stance.csv:", sum(1 for _ in open("data/MGTAB/labels_stance.csv")))

Рёбра (первые 5 строк):
    0        1        2        3        4        5        6        7        \
0        0        1        2        3        4        5        6        7   
1        0        0        0        0        0        0        0        0   
2     4665     1506     1414     1978      120     4879     3818     8846   

   8        9        ...  1700098  1700099  1700100  1700101  1700102  \
0        8        9  ...  1700098  1700099  1700100  1700101  1700102   
1        0        0  ...    10144    10144    10149    10151    10154   
2     5576     8443  ...    10157    10189    10162    10165    10158   

   1700103  1700104  1700105  1700106  1700107  
0  1700103  1700104  1700105  1700106  1700107  
1    10155    10158    10159    10160    10160  
2    10174    10175    10185    10165    10188  

[3 rows x 1700108 columns]
Метки (первые 5 строк):
    0
0  0
1  1
2  2
3  0
4  0
Всего строк в edge_index.csv: 3
Всего строк в labels_stance.csv: 10200


In [19]:
import pandas as pd

DATA_ROOT = "final_data"

feat_path = f"{DATA_ROOT}/Elliptic/elliptic_txs_features.csv"
class_path = f"{DATA_ROOT}/Elliptic/elliptic_txs_classes.csv"

df_feat = pd.read_csv(feat_path, header=None, low_memory=False)
df_class = pd.read_csv(class_path)

print("=== Признаки ===")
print(f"Количество строк: {len(df_feat)}")
print(f"Первые 5 txId (столбец 0):\n{df_feat[0].head().tolist()}")
print(f"Тип данных txId: {df_feat[0].dtype}")
print(f"Уникальных txId: {df_feat[0].nunique()}")

print("\n=== Метки ===")
print(df_class['class'].value_counts())
print(f"Количество строк с известными метками (1 или 2): {len(df_class[df_class['class'].isin(['1','2'])])}")
print(f"Первые 5 txId:\n{df_class['txId'].head().tolist()}")
print(f"Тип данных txId: {df_class['txId'].dtype}")

# Проверяем пересечение
feat_ids = set(df_feat[0].astype(str))
class_ids = set(df_class['txId'].astype(str))
intersection = feat_ids & class_ids
print(f"\nПересечение txId между признаками и метками: {len(intersection)}")

=== Признаки ===
Количество строк: 203769
Первые 5 txId (столбец 0):
[230425980, 5530458, 232022460, 232438397, 230460314]
Тип данных txId: int64
Уникальных txId: 203769

=== Метки ===
class
unknown    157205
2           42019
1            4545
Name: count, dtype: int64
Количество строк с известными метками (1 или 2): 46564
Первые 5 txId:
[230425980, 5530458, 232022460, 232438397, 230460314]
Тип данных txId: int64

Пересечение txId между признаками и метками: 203769


In [38]:
 import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch_geometric.nn import SAGEConv
import pandas as pd
import numpy as np
import os
import time
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# ==================== РУЧНЫЕ МЕТРИКИ (без sklearn) ====================
def precision_recall_curve_manual(y_true, proba):
    thresholds = np.sort(np.unique(proba))
    thresholds = np.concatenate([[1.0], thresholds, [0.0]])
    precisions = []
    recalls = []
    for thresh in thresholds:
        y_pred = (proba >= thresh).astype(int)
        tp = np.sum((y_pred == 1) & (y_true == 1))
        fp = np.sum((y_pred == 1) & (y_true == 0))
        fn = np.sum((y_pred == 0) & (y_true == 1))
        precision = tp / (tp + fp) if (tp + fp) > 0 else 1.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        precisions.append(precision)
        recalls.append(recall)
    return np.array(precisions), np.array(recalls)

def auc_manual(recall, precision):
    order = np.argsort(recall)
    recall = recall[order]
    precision = precision[order]
    return np.trapz(precision, recall)

def roc_auc_manual(y_true, y_score):
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)
    n_pos = np.sum(y_true)
    n_neg = len(y_true) - n_pos
    if n_pos == 0 or n_neg == 0:
        return 0.5
    order = np.argsort(y_score)[::-1]
    y_true_sorted = y_true[order]
    tp = np.cumsum(y_true_sorted)
    fp = np.cumsum(1 - y_true_sorted)
    tpr = tp / n_pos
    fpr = fp / n_neg
    return np.trapz(tpr, fpr)

# ==================== ПАРАМЕТРЫ ====================
RANDOM_STATES = [42, 43, 44, 45, 46]
TRAIN_RATIO = 0.7
VAL_RATIO = 0.15
TEST_RATIO = 0.15

HIDDEN_DIM1 = 256
HIDDEN_DIM2 = 128
LR = 0.005
WEIGHT_DECAY = 5e-5
MAX_EPOCHS = 100
PATIENCE = 15
USE_SUBSAMPLE = True
SUBSAMPLE_SIZE = 60000

DATA_ROOT = "final_data"

# ==================== ЗАГРУЗЧИКИ ====================
def load_elliptic():
    feat_path = os.path.join(DATA_ROOT, "Elliptic", "elliptic_txs_features.csv")
    edge_path = os.path.join(DATA_ROOT, "Elliptic", "elliptic_txs_edgelist.csv")
    class_path = os.path.join(DATA_ROOT, "Elliptic", "elliptic_txs_classes.csv")

    df_feat = pd.read_csv(feat_path, header=None, low_memory=False)
    df_feat.columns = ['txId', 'time_step'] + [f'f{i}' for i in range(df_feat.shape[1]-2)]
    df_class = pd.read_csv(class_path)
    df_class = df_class[df_class['class'].isin(['1','2'])].copy()
    df_class['label'] = df_class['class'].map({'1':0, '2':1})
    # Объединение
    df = df_feat.merge(df_class[['txId', 'label']], on='txId', how='inner')

    feature_cols = [c for c in df.columns if c not in ['txId', 'time_step', 'label']]
    X = df[feature_cols].values.astype(np.float32)
    y = df['label'].values.astype(np.int64)

    # Загружаем рёбра и переиндексируем
    edges_df = pd.read_csv(edge_path, header=None)
    # Преобразуем все txId в строки для надёжного сравнения
    id_to_idx = {str(tid): i for i, tid in enumerate(df['txId'].values)}
    edge_list = []
    for _, row in edges_df.iterrows():
        src = str(row[0])
        dst = str(row[1])
        if src in id_to_idx and dst in id_to_idx:
            edge_list.append([id_to_idx[src], id_to_idx[dst]])
    if edge_list:
        edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
    else:
        edge_index = torch.empty((2, 0), dtype=torch.long)

    return torch.tensor(X, dtype=torch.float32), edge_index, torch.tensor(y, dtype=torch.long)


def load_amazon():
    feat_path = os.path.join(DATA_ROOT, "Amazon", "Amazon_full.csv")
    edge_path = os.path.join(DATA_ROOT, "Amazon", "Amazon_net_uvu_edges.csv")
    df_feat = pd.read_csv(feat_path, header=None, skiprows=1, low_memory=False)
    x = torch.tensor(df_feat.iloc[:, :-1].values.astype(np.float32))
    y = torch.tensor(df_feat.iloc[:, -1].values.astype(np.float32), dtype=torch.long)
    edges = pd.read_csv(edge_path)[['row', 'col']].values
    edge_index = torch.tensor(edges.T, dtype=torch.long)
    return x, edge_index, y

def load_yelp():
    feat_path = os.path.join(DATA_ROOT, "YelpChi", "YelpChi_full.csv")
    edge_path = os.path.join(DATA_ROOT, "YelpChi", "YelpChi_net_rur_edges.csv")
    df_feat = pd.read_csv(feat_path)
    x = torch.tensor(df_feat.iloc[:, :-1].values.astype(np.float32))
    y = torch.tensor(df_feat.iloc[:, -1].values.astype(np.float32), dtype=torch.long)
    edges = pd.read_csv(edge_path)[['row', 'col']].values
    edge_index = torch.tensor(edges.T, dtype=torch.long)
    return x, edge_index, y

def load_mgtab():
    feat_path = os.path.join(DATA_ROOT, "MGTAB", "features.csv")
    label_path = os.path.join(DATA_ROOT, "MGTAB", "labels_bot.csv")
    edge_path = os.path.join(DATA_ROOT, "MGTAB", "edge_index.csv")

    # Признаки – все строки, без пропуска
    df_feat = pd.read_csv(feat_path, header=None, low_memory=False)
    x = torch.tensor(df_feat.values.astype(np.float32))

    # Метки
    df_labels = pd.read_csv(label_path, header=None)
    y = torch.tensor(df_labels[0].values, dtype=torch.long)

    # Рёбра
    df_edges = pd.read_csv(edge_path, header=None, low_memory=False)
    sources = df_edges.iloc[1].values.astype(np.int64)
    targets = df_edges.iloc[2].values.astype(np.int64)
    edge_index = torch.tensor([sources, targets], dtype=torch.long)

    num_nodes_edge = edge_index.max().item() + 1
    if x.size(0) != num_nodes_edge:
        # Дополняем или обрезаем до num_nodes_edge (обычно 10200)
        if x.size(0) < num_nodes_edge:
            pad = torch.zeros((num_nodes_edge - x.size(0), x.size(1)), dtype=torch.float32)
            x = torch.cat([x, pad], dim=0)
        else:
            x = x[:num_nodes_edge]
    if y.size(0) != num_nodes_edge:
        if y.size(0) < num_nodes_edge:
            pad_y = torch.full((num_nodes_edge - y.size(0),), 0, dtype=torch.long)
            y = torch.cat([y, pad_y], dim=0)
        else:
            y = y[:num_nodes_edge]

    print(f"  Узлов: {x.size(0)}, признаков: {x.size(1)}, рёбер: {edge_index.size(1)}, меток: {y.size(0)}")
    return x, edge_index, y


GRAPH_DATASETS = {
    "MGTAB": load_mgtab,
    "Elliptic (Bitcoin)": load_elliptic,
    "Amazon Reviews": load_amazon,
    "Yelp Reviews": load_yelp,

}

# ==================== МОДЕЛЬ ====================
class GraphSAGE(torch.nn.Module):
    def __init__(self, in_channels, hidden_dim1=256, hidden_dim2=128):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_dim1)
        self.bn1 = nn.BatchNorm1d(hidden_dim1)
        self.conv2 = SAGEConv(hidden_dim1, hidden_dim2)
        self.bn2 = nn.BatchNorm1d(hidden_dim2)
        self.lin = nn.Linear(hidden_dim2, 1)
    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = F.relu(x)
        x = self.lin(x)
        return x

# ==================== ВСПОМОГАТЕЛЬНЫЕ ====================
def split_nodes(y, train_ratio, val_ratio, test_ratio, random_state):
    np.random.seed(random_state)
    n = len(y)
    perm = np.random.permutation(n)
    train_end = int(n * train_ratio)
    val_end = int(n * (train_ratio + val_ratio))
    return perm[:train_end], perm[train_end:val_end], perm[val_end:]

def compute_metrics(y_true, y_pred_proba):
    y_true_np = y_true.cpu().numpy() if torch.is_tensor(y_true) else y_true
    y_pred_proba_np = y_pred_proba.cpu().numpy() if torch.is_tensor(y_pred_proba) else y_pred_proba
    prec, rec = precision_recall_curve_manual(y_true_np, y_pred_proba_np)
    auprc = auc_manual(rec, prec)
    auroc = roc_auc_manual(y_true_np, y_pred_proba_np)
    K = y_true_np.sum()
    if K > 0:
        top_k_idx = np.argsort(y_pred_proba_np)[-K:]
        prec_at_k = y_true_np[top_k_idx].mean()
    else:
        prec_at_k = np.nan
    return auprc, auroc, prec_at_k

# ==================== ОСНОВНОЙ ЭКСПЕРИМЕНТ ====================
def run_graphsage_experiment():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Используется устройство: {device}")
    results = []
    for name, loader in GRAPH_DATASETS.items():
        print(f"\n{'='*60}\nЗапуск GraphSAGE на датасете: {name}\n{'='*60}")
        try:
            x, edge_index, y = loader()
            print(f"  Узлов: {x.size(0)}, признаков: {x.size(1)}, рёбер: {edge_index.size(1)}, меток: {y.size(0)}")
        except Exception as e:
            print(f"  Ошибка загрузки: {e}")
            continue
        if len(torch.unique(y)) < 2:
            print("  Пропускаем: только один класс")
            continue
        auprc_list, auroc_list, pk_list = [], [], []
        train_times, test_times = [], []
        for seed in tqdm(RANDOM_STATES, desc="Seeds"):
            torch.manual_seed(seed)
            np.random.seed(seed)
            train_idx, val_idx, test_idx = split_nodes(y, TRAIN_RATIO, VAL_RATIO, TEST_RATIO, seed)
            if USE_SUBSAMPLE and len(train_idx) > SUBSAMPLE_SIZE:
                sub_idx = np.random.choice(train_idx, size=SUBSAMPLE_SIZE, replace=False)
                train_idx = sub_idx
                print(f"    Подвыборка тренировочных узлов: {len(train_idx)}")
            model = GraphSAGE(x.size(1), HIDDEN_DIM1, HIDDEN_DIM2).to(device)
            y_train = y[train_idx]
            pos = (y_train == 1).sum().item()
            neg = (y_train == 0).sum().item()
            scale_pos_weight = min(neg / pos, 500) if pos > 0 else 1.0
            pos_weight = torch.tensor([scale_pos_weight], device=device)
            criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
            optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
            scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)
            x_dev = x.to(device)
            edge_index_dev = edge_index.to(device)
            y_dev = y.to(device)
            best_val_auprc = 0
            best_model_state = None
            wait = 0
            start_train = time.time()
            for epoch in range(1, MAX_EPOCHS+1):
                model.train()
                optimizer.zero_grad()
                out = model(x_dev, edge_index_dev).squeeze()
                loss = criterion(out[train_idx], y_dev[train_idx].float())
                loss.backward()
                optimizer.step()
                scheduler.step()
                model.eval()
                with torch.no_grad():
                    out_val = model(x_dev, edge_index_dev).squeeze()
                    proba_val = torch.sigmoid(out_val[val_idx]).cpu().numpy()
                    y_val_np = y_dev[val_idx].cpu().numpy()
                    prec, rec = precision_recall_curve_manual(y_val_np, proba_val)
                    val_auprc = auc_manual(rec, prec)
                if val_auprc > best_val_auprc:
                    best_val_auprc = val_auprc
                    best_model_state = model.state_dict().copy()
                    wait = 0
                else:
                    wait += 1
                    if wait >= PATIENCE:
                        break
            train_time = time.time() - start_train
            model.load_state_dict(best_model_state)
            model.eval()
            start_test = time.time()
            with torch.no_grad():
                out_test = model(x_dev, edge_index_dev).squeeze()
                proba_test = torch.sigmoid(out_test[test_idx]).cpu().numpy()
            test_time = time.time() - start_test
            auprc, auroc, pk = compute_metrics(y_dev[test_idx], proba_test)
            auprc_list.append(auprc); auroc_list.append(auroc); pk_list.append(pk)
            train_times.append(train_time); test_times.append(test_time)
            print(f"    seed={seed}: AUPRC={auprc:.4f}, время={train_time:.1f}c")
        results.append({
            "Dataset": name,
            "AUPRC_mean": np.mean(auprc_list),
            "AUPRC_std": np.std(auprc_list),
            "AUROC_mean": np.mean(auroc_list),
            "AUROC_std": np.std(auroc_list),
            "Prec@K_mean": np.nanmean(pk_list),
            "Prec@K_std": np.nanstd(pk_list),
            "Train_time_avg": np.mean(train_times),
            "Test_time_avg": np.mean(test_times),
        })
        print(f"  Итог {name}: AUPRC = {np.mean(auprc_list):.4f} ± {np.std(auprc_list):.4f}")
    if results:
        df_res = pd.DataFrame(results)
        print("\n" + "="*80)
        print("РЕЗУЛЬТАТЫ GRAPHSAGE")
        print("="*80)
        print(df_res.round(4).to_string(index=False))
        df_res.to_csv("graphsage_results.csv", index=False)
        print("\nСохранено в graphsage_results.csv")
    else:
        print("Нет результатов для сохранения")

if __name__ == "__main__":
    run_graphsage_experiment()

Используется устройство: cpu

Запуск GraphSAGE на датасете: MGTAB
  Узлов: 10199, признаков: 788, рёбер: 1700108, меток: 10199
  Узлов: 10199, признаков: 788, рёбер: 1700108, меток: 10199


Seeds:  20%|██        | 1/5 [04:36<18:27, 276.81s/it]

    seed=42: AUPRC=0.8027, время=275.5c


Seeds:  40%|████      | 2/5 [09:49<14:53, 297.70s/it]

    seed=43: AUPRC=0.8519, время=310.9c


...

    seed=44: AUPRC=0.6505, время=32.2c


Seeds:  80%|████████  | 4/5 [01:16<00:19, 19.64s/it]

    seed=45: AUPRC=0.5573, время=14.5c


Seeds: 100%|██████████| 5/5 [01:32<00:00, 18.43s/it]

    seed=46: AUPRC=0.5314, время=15.2c
  Итог Yelp Reviews: AUPRC = 0.5651 ± 0.0539

РЕЗУЛЬТАТЫ GRAPHSAGE
           Dataset  AUPRC_mean  AUPRC_std  AUROC_mean  AUROC_std  Prec@K_mean  Prec@K_std  Train_time_avg  Test_time_avg
             MGTAB      0.8260     0.0227      0.9355     0.0064       0.7727      0.0214        314.1355         1.3577
Elliptic (Bitcoin)      0.9964     0.0014      0.9831     0.0016       0.9863      0.0008         19.9570         0.0501
    Amazon Reviews      0.8125     0.0353      0.9704     0.0061       0.7989      0.0285         80.7430         0.4182
      Yelp Reviews      0.5651     0.0539      0.8679     0.0265       0.5321      0.0431         18.2209         0.0487

Сохранено в graphsage_results.csv


In [48]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch_geometric.nn import GCNConv, GATConv
import pandas as pd
import numpy as np
import os
import time
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# ==================== РУЧНЫЕ МЕТРИКИ (без sklearn) ====================
def precision_recall_curve_manual(y_true, proba):
    thresholds = np.sort(np.unique(proba))
    thresholds = np.concatenate([[1.0], thresholds, [0.0]])
    precisions = []
    recalls = []
    for thresh in thresholds:
        y_pred = (proba >= thresh).astype(int)
        tp = np.sum((y_pred == 1) & (y_true == 1))
        fp = np.sum((y_pred == 1) & (y_true == 0))
        fn = np.sum((y_pred == 0) & (y_true == 1))
        precision = tp / (tp + fp) if (tp + fp) > 0 else 1.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        precisions.append(precision)
        recalls.append(recall)
    return np.array(precisions), np.array(recalls)

def auc_manual(recall, precision):
    order = np.argsort(recall)
    recall = recall[order]
    precision = precision[order]
    return np.trapz(precision, recall)

def roc_auc_manual(y_true, y_score):
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)
    n_pos = np.sum(y_true)
    n_neg = len(y_true) - n_pos
    if n_pos == 0 or n_neg == 0:
        return 0.5
    order = np.argsort(y_score)[::-1]
    y_true_sorted = y_true[order]
    tp = np.cumsum(y_true_sorted)
    fp = np.cumsum(1 - y_true_sorted)
    tpr = tp / n_pos
    fpr = fp / n_neg
    return np.trapz(tpr, fpr)

def compute_metrics_manual(y_true, y_pred_proba):
    y_true_np = y_true.cpu().numpy() if torch.is_tensor(y_true) else y_true
    y_pred_proba_np = y_pred_proba.cpu().numpy() if torch.is_tensor(y_pred_proba) else y_pred_proba
    prec, rec = precision_recall_curve_manual(y_true_np, y_pred_proba_np)
    auprc = auc_manual(rec, prec)
    auroc = roc_auc_manual(y_true_np, y_pred_proba_np)
    K = y_true_np.sum()
    if K > 0:
        top_k_idx = np.argsort(y_pred_proba_np)[-K:]
        prec_at_k = y_true_np[top_k_idx].mean()
    else:
        prec_at_k = np.nan
    return auprc, auroc, prec_at_k

# ==================== ПАРАМЕТРЫ ЭКСПЕРИМЕНТА ====================
RANDOM_STATES = [42, 43, 44, 45, 46]
TRAIN_RATIO = 0.7
VAL_RATIO = 0.15
TEST_RATIO = 0.15

HIDDEN_DIM1 = 256
HIDDEN_DIM2 = 128
LR = 0.005
WEIGHT_DECAY = 5e-5
MAX_EPOCHS = 100
PATIENCE = 15
USE_SUBSAMPLE = True
SUBSAMPLE_SIZE = 20000
GAT_HEADS = 8

DATA_ROOT = "final_data"

# ==================== ЗАГРУЗЧИКИ ДАННЫХ ====================
def load_elliptic():
    feat_path = os.path.join(DATA_ROOT, "Elliptic", "elliptic_txs_features.csv")
    edge_path = os.path.join(DATA_ROOT, "Elliptic", "elliptic_txs_edgelist.csv")
    class_path = os.path.join(DATA_ROOT, "Elliptic", "elliptic_txs_classes.csv")

    df_feat = pd.read_csv(feat_path, header=None, low_memory=False)
    df_feat.columns = ['txId', 'time_step'] + [f'f{i}' for i in range(df_feat.shape[1]-2)]
    df_class = pd.read_csv(class_path)
    df_class = df_class[df_class['class'].isin(['1','2'])].copy()
    df_class['label'] = df_class['class'].map({'1':0, '2':1})
    df = df_feat.merge(df_class[['txId', 'label']], on='txId', how='inner')
    feature_cols = [c for c in df.columns if c not in ['txId', 'time_step', 'label']]
    X = df[feature_cols].values.astype(np.float32)
    y = df['label'].values.astype(np.int64)

    edges_df = pd.read_csv(edge_path, header=None)
    id_to_idx = {str(tid): i for i, tid in enumerate(df['txId'].values)}
    edge_list = []
    for _, row in edges_df.iterrows():
        src = str(row[0]); dst = str(row[1])
        if src in id_to_idx and dst in id_to_idx:
            edge_list.append([id_to_idx[src], id_to_idx[dst]])
    if edge_list:
        edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
    else:
        edge_index = torch.empty((2,0), dtype=torch.long)
    return torch.tensor(X, dtype=torch.float32), edge_index, torch.tensor(y, dtype=torch.long)

def load_amazon():
    feat_path = os.path.join(DATA_ROOT, "Amazon", "Amazon_full.csv")
    edge_path = os.path.join(DATA_ROOT, "Amazon", "Amazon_net_uvu_edges.csv")
    df_feat = pd.read_csv(feat_path, header=None, skiprows=1, low_memory=False)
    X = df_feat.iloc[:, :-1].values.astype(np.float32)
    y = df_feat.iloc[:, -1].values.astype(np.int64)
    edges = pd.read_csv(edge_path)[['row', 'col']].values
    edge_index = torch.tensor(edges.T, dtype=torch.long)
    return torch.tensor(X, dtype=torch.float32), edge_index, torch.tensor(y, dtype=torch.long)

def load_yelp():
    feat_path = os.path.join(DATA_ROOT, "YelpChi", "YelpChi_full.csv")
    edge_path = os.path.join(DATA_ROOT, "YelpChi", "YelpChi_net_rur_edges.csv")
    df_feat = pd.read_csv(feat_path)
    X = df_feat.iloc[:, :-1].values.astype(np.float32)
    y = df_feat.iloc[:, -1].values.astype(np.int64)
    edges = pd.read_csv(edge_path)[['row', 'col']].values
    edge_index = torch.tensor(edges.T, dtype=torch.long)
    return torch.tensor(X, dtype=torch.float32), edge_index, torch.tensor(y, dtype=torch.long)

def load_mgtab():
    feat_path = os.path.join(DATA_ROOT, "MGTAB", "features.csv")
    label_path = os.path.join(DATA_ROOT, "MGTAB", "labels_bot.csv")
    edge_path = os.path.join(DATA_ROOT, "MGTAB", "edge_index.csv")
    df_feat = pd.read_csv(feat_path, header=None, low_memory=False)
    X = df_feat.values.astype(np.float32)
    df_labels = pd.read_csv(label_path, header=None)
    y = df_labels[0].values.astype(np.int64)
    df_edges = pd.read_csv(edge_path, header=None, low_memory=False)
    sources = df_edges.iloc[1].values.astype(np.int64)
    targets = df_edges.iloc[2].values.astype(np.int64)
    edge_index = torch.tensor([sources, targets], dtype=torch.long)
    num_nodes_edge = edge_index.max().item() + 1
    if X.shape[0] != num_nodes_edge:
        if X.shape[0] < num_nodes_edge:
            pad = np.zeros((num_nodes_edge - X.shape[0], X.shape[1]), dtype=np.float32)
            X = np.vstack([X, pad])
        else:
            X = X[:num_nodes_edge]
    if len(y) != num_nodes_edge:
        if len(y) < num_nodes_edge:
            pad_y = np.zeros(num_nodes_edge - len(y), dtype=np.int64)
            y = np.concatenate([y, pad_y])
        else:
            y = y[:num_nodes_edge]
    return torch.tensor(X, dtype=torch.float32), edge_index, torch.tensor(y, dtype=torch.long)

GRAPH_DATASETS = {
    "MGTAB": load_mgtab,
    "Elliptic (Bitcoin)": load_elliptic,
    "Amazon Reviews": load_amazon,
    "Yelp Reviews": load_yelp,
}

# ==================== МОДЕЛИ ====================
class GCN(torch.nn.Module):
    def __init__(self, in_channels, hidden1=256, hidden2=128):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden1)
        self.bn1 = nn.BatchNorm1d(hidden1)
        self.conv2 = GCNConv(hidden1, hidden2)
        self.bn2 = nn.BatchNorm1d(hidden2)
        self.lin = nn.Linear(hidden2, 1)
    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = F.relu(x)
        x = self.lin(x)
        return x

class GAT(torch.nn.Module):
    def __init__(self, in_channels, hidden1=256, hidden2=128, heads=8):
        super().__init__()
        self.conv1 = GATConv(in_channels, hidden1 // heads, heads=heads, concat=True)
        self.bn1 = nn.BatchNorm1d(hidden1)
        self.conv2 = GATConv(hidden1, hidden2, heads=1, concat=False)
        self.bn2 = nn.BatchNorm1d(hidden2)
        self.lin = nn.Linear(hidden2, 1)
    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.relu(x)
        x = F.dropout(x, p=0.3, training=self.training)
        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = F.relu(x)
        x = self.lin(x)
        return x

# ==================== ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ ====================
def split_nodes(y, train_ratio, val_ratio, test_ratio, random_state):
    np.random.seed(random_state)
    n = len(y)
    perm = np.random.permutation(n)
    train_end = int(n * train_ratio)
    val_end = int(n * (train_ratio + val_ratio))
    return perm[:train_end], perm[train_end:val_end], perm[val_end:]

# ==================== ФУНКЦИЯ ОБУЧЕНИЯ ДЛЯ GNN ====================
def run_gnn_experiment(model_class, model_name):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Используется устройство: {device}")
    results = []
    for name, loader in GRAPH_DATASETS.items():
        print(f"\n{'='*60}\nЗапуск {model_name} на датасете: {name}\n{'='*60}")
        try:
            x, edge_index, y = loader()
            print(f"  Узлов: {x.size(0)}, признаков: {x.size(1)}, рёбер: {edge_index.size(1)}, меток: {y.size(0)}")
        except Exception as e:
            print(f"  Ошибка загрузки: {e}")
            continue
        if len(torch.unique(y)) < 2:
            print("  Пропускаем: только один класс")
            continue

        auprc_list, auroc_list, pk_list = [], [], []
        train_times, test_times = [], []
        for seed in tqdm(RANDOM_STATES, desc="Seeds"):
            torch.manual_seed(seed)
            np.random.seed(seed)
            train_idx, val_idx, test_idx = split_nodes(y, TRAIN_RATIO, VAL_RATIO, TEST_RATIO, seed)
            if USE_SUBSAMPLE and len(train_idx) > SUBSAMPLE_SIZE:
                sub_idx = np.random.choice(train_idx, size=SUBSAMPLE_SIZE, replace=False)
                train_idx = sub_idx
                print(f"    Подвыборка тренировочных узлов: {len(train_idx)}")

            if model_class == GAT:
                model = GAT(x.size(1), HIDDEN_DIM1, HIDDEN_DIM2, heads=GAT_HEADS).to(device)
            else:
                model = GCN(x.size(1), HIDDEN_DIM1, HIDDEN_DIM2).to(device)

            y_train = y[train_idx]
            pos = (y_train == 1).sum().item()
            neg = (y_train == 0).sum().item()
            scale_pos_weight = min(neg / pos, 500) if pos > 0 else 1.0
            pos_weight = torch.tensor([scale_pos_weight], device=device)
            criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
            optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
            scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)

            x_dev = x.to(device)
            edge_index_dev = edge_index.to(device)
            y_dev = y.to(device)
            best_val_auprc = 0
            best_model_state = None
            wait = 0
            start_train = time.time()
            for epoch in range(1, MAX_EPOCHS+1):
                model.train()
                optimizer.zero_grad()
                out = model(x_dev, edge_index_dev).squeeze()
                loss = criterion(out[train_idx], y_dev[train_idx].float())
                loss.backward()
                optimizer.step()
                scheduler.step()
                # Валидация
                model.eval()
                with torch.no_grad():
                    out_val = model(x_dev, edge_index_dev).squeeze()
                    proba_val = torch.sigmoid(out_val[val_idx]).cpu().numpy()
                    y_val_np = y_dev[val_idx].cpu().numpy()
                    prec, rec = precision_recall_curve_manual(y_val_np, proba_val)
                    val_auprc = auc_manual(rec, prec)
                if val_auprc > best_val_auprc:
                    best_val_auprc = val_auprc
                    best_model_state = model.state_dict().copy()
                    wait = 0
                else:
                    wait += 1
                    if wait >= PATIENCE:
                        break
            train_time = time.time() - start_train
            # Тестирование с лучшей модельой
            model.load_state_dict(best_model_state)
            model.eval()
            start_test = time.time()
            with torch.no_grad():
                out_test = model(x_dev, edge_index_dev).squeeze()
                proba_test = torch.sigmoid(out_test[test_idx]).cpu().numpy()
            test_time = time.time() - start_test
            auprc, auroc, pk = compute_metrics_manual(y_dev[test_idx], proba_test)
            auprc_list.append(auprc); auroc_list.append(auroc); pk_list.append(pk)
            train_times.append(train_time); test_times.append(test_time)
            print(f"    seed={seed}: AUPRC={auprc:.4f}, время={train_time:.1f}c")

        results.append({
            "Dataset": name,
            "AUPRC_mean": np.mean(auprc_list),
            "AUPRC_std": np.std(auprc_list),
            "AUROC_mean": np.mean(auroc_list),
            "AUROC_std": np.std(auroc_list),
            "Prec@K_mean": np.nanmean(pk_list),
            "Prec@K_std": np.nanstd(pk_list),
            "Train_time_avg": np.mean(train_times),
            "Test_time_avg": np.mean(test_times),
        })
        print(f"  Итог {name}: AUPRC = {np.mean(auprc_list):.4f} ± {np.std(auprc_list):.4f}")

    df_res = pd.DataFrame(results)
    print("\n" + "="*80)
    print(f"РЕЗУЛЬТАТЫ {model_name}")
    print("="*80)
    print(df_res.round(4).to_string(index=False))
    df_res.to_csv(f"{model_name.lower()}_results.csv", index=False)
    print(f"\nСохранено в {model_name.lower()}_results.csv")
    return df_res

if __name__ == "__main__":
    run_gnn_experiment(GCN, "GCN")
    run_gnn_experiment(GAT, "GAT")

Используется устройство: cpu

Запуск GCN на датасете: MGTAB
  Узлов: 10199, признаков: 788, рёбер: 1700108, меток: 10199


Seeds:  20%|██        | 1/5 [01:49<07:18, 109.59s/it]

    seed=42: AUPRC=0.3299, время=109.1c


Seeds:  40%|████      | 2/5 [02:14<03:00, 60.03s/it] 

    seed=43: AUPRC=0.2685, время=24.9c


...

    seed=44: AUPRC=0.6542, время=68.4c
    Подвыборка тренировочных узлов: 20000


Seeds:  80%|████████  | 4/5 [04:32<01:08, 68.40s/it]

    seed=45: AUPRC=0.6643, время=68.7c
    Подвыборка тренировочных узлов: 20000


Seeds: 100%|██████████| 5/5 [05:42<00:00, 68.46s/it]

    seed=46: AUPRC=0.6569, время=69.3c
  Итог Yelp Reviews: AUPRC = 0.6659 ± 0.0230

РЕЗУЛЬТАТЫ GAT
           Dataset  AUPRC_mean  AUPRC_std  AUROC_mean  AUROC_std  Prec@K_mean  Prec@K_std  Train_time_avg  Test_time_avg
             MGTAB      0.2908     0.0138      0.5216     0.0197       0.2810      0.0156         54.7695         0.5945
Elliptic (Bitcoin)      0.9969     0.0004      0.9812     0.0012       0.9851      0.0011         31.0799         0.0651
    Amazon Reviews      0.5599     0.0449      0.9111     0.0143       0.5385      0.0216        214.1710         0.9881
      Yelp Reviews      0.6659     0.0230      0.9132     0.0056       0.6250      0.0169         68.1450         0.1106

Сохранено в gat_results.csv
